In this notebook, we showcase how to use the improve retrieval performance using chunking.

In [1]:
import numpy as np
import torch
from transformers import pipeline

from kvpress import (
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    apply_per_layer_compression,
)

/mount/home/setup/.cache/pypoetry/virtualenvs/kvpress-PV_RntMw-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the pipeline and data

In [2]:
# Load pipeline

device = "cuda:0"
ckpt = "meta-llama/Meta-Llama-3.1-8B-Instruct"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, torch_dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  8.94it/s]
Device set to use cuda:0


In [3]:
import datasets 

df = datasets.load_dataset("simonjegou/ruler", "4096")["test"].to_pandas()
df = df.loc[df["task"] == "niah_single_3"].reset_index(drop=True)

# Use the pipeline with a press

In [4]:
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.DEBUG)

In [5]:
# Pick a press with a compression ratio, you can run the following cells with different presses
compression_ratio = 0.3
press = KnormPress(compression_ratio)

In [6]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=press, 
                   max_new_token=1000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

DEBUG:kvpress.pipeline:Context Length: 3774
DEBUG:kvpress.pipeline:Compressed Context Length: 2641


Question:   What is the special magic uuid for amused-quart mentioned in the provided text? 
Answer:     1ff49b78-8946-4e85-b59c-de66bacfb3d0
Prediction: The special magic uuid for amused-quart is: 1ff49b78-8946-4e85-b59d-e6b9b3c3d0
Correctly predicted: False


In [7]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=press, 
                   chunk_size=1000,
                   max_new_token=1000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

DEBUG:kvpress.pipeline:Context Length: 3774
DEBUG:kvpress.pipeline:Compressed Context Length: 2641


Question:   What is the special magic uuid for amused-quart mentioned in the provided text? 
Answer:     1ff49b78-8946-4e85-b59c-de66bacfb3d0
Prediction: The special magic uuid for amused-quart is: 1ff49b78-8946-4e85-b59c-de66bacfb3d0
Correctly predicted: True


In [8]:
from kvpress import BasePress

In [11]:
#BasePress??

In [16]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=KnormPress(compression_ratio=0.3, prune_independently=True), 
                   chunk_size=1000,
                   max_new_token=1000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

DEBUG:kvpress.pipeline:Context Length: 3774
DEBUG:kvpress.pipeline:Compressed Context Length: 2641


Question:   What is the special magic uuid for amused-quart mentioned in the provided text? 
Answer:     1ff49b78-8946-4e85-b59c-de66bacfb3d0
Prediction: The special magic uuid for amused-quart is: 1ff49b78-8946-4e85-b59c-de66bacfb3d0
Correctly predicted: True


In [21]:
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.CRITICAL)

In [22]:
from tqdm import tqdm

correct = []
# Run the pipeline on a single question
for idx in tqdm(range(len(df))):
    context = df.iloc[idx]["context"] 
    question = df.iloc[idx]["question"] 
    true_answer = df.iloc[idx]["answer"][0]
    
    pred_answer = pipe(context, question=question, 
                       press=KnormPress(compression_ratio=0.3, prune_independently=False), 
                       chunk_size=1000,
                       max_new_token=100)["answer"]
    correct.append(true_answer in pred_answer)

100%|██████████| 500/500 [12:25<00:00,  1.49s/it]


In [23]:
np.mean(correct)

np.float64(0.332)

In [24]:
from tqdm import tqdm

correct = []
# Run the pipeline on a single question
for idx in tqdm(range(len(df))):
    context = df.iloc[idx]["context"] 
    question = df.iloc[idx]["question"] 
    true_answer = df.iloc[idx]["answer"][0]
    
    pred_answer = pipe(context, question=question, 
                       press=KnormPress(compression_ratio=0.3, prune_independently=False), 
                       chunk_size=100000,
                       max_new_token=100)["answer"]
    correct.append(true_answer in pred_answer)

100%|██████████| 500/500 [11:46<00:00,  1.41s/it]


In [25]:
np.mean(correct)

np.float64(0.202)

In [28]:
from tqdm import tqdm

for chunk_size in [500, 1000, 1500, 2500, 3500, 4500]:
    correct = []
    # Run the pipeline on a single question
    for idx in tqdm(range(len(df))):
        context = df.iloc[idx]["context"] 
        question = df.iloc[idx]["question"] 
        true_answer = df.iloc[idx]["answer"][0]
        
        pred_answer = pipe(context, question=question, 
                           press=KnormPress(compression_ratio=0.3, prune_independently=False), 
                           chunk_size=chunk_size,
                           max_new_token=100)["answer"]
        correct.append(true_answer in pred_answer)
    print(chunk_size, np.mean(correct))

100%|██████████| 500/500 [13:05<00:00,  1.57s/it]


500 0.296


100%|██████████| 500/500 [12:24<00:00,  1.49s/it]


1000 0.332


100%|██████████| 500/500 [12:20<00:00,  1.48s/it]


...

2500 0.278


100%|██████████| 500/500 [12:16<00:00,  1.47s/it]


3500 0.19


100%|██████████| 500/500 [11:47<00:00,  1.41s/it]

4500 0.202


In [36]:
from tqdm import tqdm

for chunk_size in [500, 1000, 1500, 2500, 3500, 4500]:
    correct = []
    # Run the pipeline on a single question
    for idx in tqdm(range(len(df))):
        context = df.iloc[idx]["context"] 
        question = df.iloc[idx]["question"] 
        true_answer = df.iloc[idx]["answer"][0]
        
        pred_answer = pipe(context, question=question, 
                           press=KnormPress(compression_ratio=0.3, prune_independently=True), 
                           chunk_size=chunk_size,
                           max_new_token=100)["answer"]
        correct.append(true_answer in pred_answer)
    print(chunk_size, np.mean(correct))

100%|██████████| 500/500 [15:12<00:00,  1.83s/it]


500 0.282


100%|██████████| 500/500 [13:04<00:00,  1.57s/it]


1000 0.322


100%|██████████| 500/500 [12:24<00:00,  1.49s/it]


...

2500 0.282


100%|██████████| 500/500 [13:10<00:00,  1.58s/it]


3500 0.19


100%|██████████| 500/500 [14:34<00:00,  1.75s/it]

4500 0.202


In [29]:
from tqdm import tqdm

for chunk_size in [500, 1000, 1500, 2500, 3500, 4500]:
    correct = []
    # Run the pipeline on a single question
    for idx in tqdm(range(len(df))):
        context = df.iloc[idx]["context"] 
        question = df.iloc[idx]["question"] 
        true_answer = df.iloc[idx]["answer"][0]
        
        pred_answer = pipe(context, question=question, 
                           press=ExpectedAttentionPress(compression_ratio=0.3, prune_independently=False), 
                           chunk_size=chunk_size,
                           max_new_token=100)["answer"]
        correct.append(true_answer in pred_answer)
    print(chunk_size, np.mean(correct))

100%|██████████| 500/500 [18:18<00:00,  2.20s/it]


500 0.218


100%|██████████| 500/500 [15:23<00:00,  1.85s/it]


1000 0.302


100%|██████████| 500/500 [14:35<00:00,  1.75s/it]


...

2500 0.326


100%|██████████| 500/500 [14:20<00:00,  1.72s/it]


3500 0.192


100%|██████████| 500/500 [13:56<00:00,  1.67s/it]

4500 0.256


In [33]:
from transformers import DynamicCache

for chunk_size in [500, 1000, 1500, 2000, 5000, 10000]:
    cache = DynamicCache()
    
    pred_answer = pipe(context, question=question, 
                       press=ExpectedAttentionPress(compression_ratio=0.3, prune_independently=False), 
                       chunk_size=chunk_size,
                       max_new_token=100,
                       cache=cache)["answer"]
    print(cache.get_seq_length())

2643
2643
2643
2643
2643
2643


In [35]:
from transformers import DynamicCache

for chunk_size in [500, 1000, 1500, 2000, 5000, 10000]:
    cache = DynamicCache()
    
    pred_answer = pipe(context, question=question, 
                       press=ExpectedAttentionPress(compression_ratio=0.9, prune_independently=False), 
                       chunk_size=chunk_size,
                       max_new_token=100,
                       cache=cache)["answer"]
    print(cache.get_seq_length())

370
374
375
376
377
377


In [34]:
from transformers import DynamicCache

for chunk_size in [500, 1000, 1500, 2000, 5000, 10000]:
    cache = DynamicCache()
    
    pred_answer = pipe(context, question=question, 
                       press=ExpectedAttentionPress(compression_ratio=0.3, prune_independently=True), 
                       chunk_size=chunk_size,
                       max_new_token=100,
                       cache=cache)["answer"]
    print(cache.get_seq_length())

2643
2643
2643
2643
2643
2643
